# Lab 3.3 - Customize Output with Sampling Parameters (Solution)
**Day 3 - Session 2**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kpassoubady/gen-ai-companion-intuite/blob/main/day3/lab3_3_sampling/solution/lab3_3_sampling.ipynb)

**Goal:** Generate text with a small GPT-2 style model and see how the
decoding parameters (temperature, top-k, top-p, beam search) change the
output. Same model, same prompt, different knobs.

This solution has the full working code. Everything runs offline on a Colab
CPU runtime in a couple of minutes; no API key and no GPU needed.

In [ ]:
# Scoped install for this lab (safe to re-run; preinstalled on Colab).
%pip install -q transformers torch

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "distilgpt2"  # a small distilled GPT-2, fast on CPU
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model.eval()

PROMPT = "The best way to learn programming is"
input_ids = tokenizer(PROMPT, return_tensors="pt").input_ids
print("Model loaded. Prompt:", repr(PROMPT))

## 1. Greedy decoding: the baseline
Greedy decoding always picks the single most likely next token. It is
deterministic, so the output never changes for the same prompt. It can also
get repetitive.

In [ ]:
greedy_ids = model.generate(
    input_ids,
    max_new_tokens=40,
    do_sample=False,
    pad_token_id=tokenizer.eos_token_id,
)
print(tokenizer.decode(greedy_ids[0], skip_special_tokens=True))

## 2. Temperature: the creativity dial
Temperature reshapes the probability distribution before sampling. Low
temperature (0.2) stays close to greedy and safe. High temperature (1.3)
flattens the distribution and picks unlikely tokens more often, so the text
gets more varied and less predictable.

In [ ]:
def generate_with_temperature(temp):
    torch.manual_seed(0)  # fix the seed so only temperature changes
    output = model.generate(
        input_ids,
        max_new_tokens=40,
        do_sample=True,
        temperature=temp,
        top_k=0,
        top_p=1.0,
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(output[0], skip_special_tokens=True)


for t in (0.2, 0.7, 1.3):
    print(f"\n--- temperature = {t} ---")
    print(generate_with_temperature(t))

## 3. Top-k sampling
Top-k keeps only the k most likely next tokens and samples from those. A
small k (10) stays focused; a large k (100) allows more variety.

In [ ]:
def generate_with_top_k(k):
    torch.manual_seed(0)
    output = model.generate(
        input_ids,
        max_new_tokens=40,
        do_sample=True,
        top_k=k,
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(output[0], skip_special_tokens=True)


for k in (10, 50, 100):
    print(f"\n--- top_k = {k} ---")
    print(generate_with_top_k(k))

## 4. Top-p (nucleus) sampling
Top-p keeps the smallest set of tokens whose probabilities add up to p, then
samples from that set. The size of the set changes token by token. Lower p
(0.5) is tighter; higher p (0.95) is looser.

In [ ]:
def generate_with_top_p(p):
    torch.manual_seed(0)
    output = model.generate(
        input_ids,
        max_new_tokens=40,
        do_sample=True,
        top_k=0,
        top_p=p,
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(output[0], skip_special_tokens=True)


for p in (0.5, 0.8, 0.95):
    print(f"\n--- top_p = {p} ---")
    print(generate_with_top_p(p))

## 5. Beam search
Beam search keeps several candidate sequences (beams) at each step and
returns the highest-scoring full sequence. It is deterministic and tends to
produce fluent, focused text, at the cost of diversity.

In [ ]:
beam_ids = model.generate(
    input_ids,
    max_new_tokens=40,
    num_beams=5,
    early_stopping=True,
    no_repeat_ngram_size=2,
    pad_token_id=tokenizer.eos_token_id,
)
print(tokenizer.decode(beam_ids[0], skip_special_tokens=True))

## 6. Side by side
The same prompt under four strategies. Greedy and beam are deterministic;
temperature, top-k, and top-p introduce randomness you can tune.

In [ ]:
def first_line(text):
    return text.replace("\n", " ").strip()


torch.manual_seed(0)
sampled = model.generate(input_ids, max_new_tokens=40, do_sample=True,
                         temperature=0.9, top_p=0.9,
                         pad_token_id=tokenizer.eos_token_id)
rows = {
    "greedy": tokenizer.decode(greedy_ids[0], skip_special_tokens=True),
    "beam (5)": tokenizer.decode(beam_ids[0], skip_special_tokens=True),
    "sampling (t=0.9, p=0.9)": tokenizer.decode(sampled[0], skip_special_tokens=True),
}
for name, text in rows.items():
    print(f"\n[{name}]\n{first_line(text)}")

## Stretch goals
1. Set a fixed `torch.manual_seed(...)` and run the top-p cell twice. Confirm
   the output is identical, then remove the seed and confirm it changes.
2. Combine `temperature=0.8` with `top_p=0.9` in a single `generate` call and
   compare the result to using each knob alone.
3. Increase `num_beams` to 8 and add `num_return_sequences=3` to inspect the
   top few beams instead of just the best one.

In [ ]:
# Stretch: temperature and top-p together, with repeatability.
torch.manual_seed(123)
combo = model.generate(
    input_ids,
    max_new_tokens=40,
    do_sample=True,
    temperature=0.8,
    top_p=0.9,
    pad_token_id=tokenizer.eos_token_id,
)
print(tokenizer.decode(combo[0], skip_special_tokens=True))

## Expected output
- Greedy and beam search print the same text every run.
- Temperature 0.2 reads close to greedy; 1.3 is noticeably more random.
- Small top-k and low top-p stay focused; larger values add variety.
- Beam search reads fluent and avoids repeated phrases.